# 📖 Notebook 4: RAG & Re-Ranking

Notebooks 1–3 used **synthetic** embeddings (sinusoids by genre/category). That was fine to teach indexes, but in real systems you:

1. Turn **real text** into vectors with a model.
2. Store those vectors in a vector DB (pgvector here).
3. At query time: vector search, then often **re-rank** with a more accurate but slower scorer.
4. For RAG: hand the top passages to an LLM as context.

This notebook walks through all four, **without any external API or ML library** — just numpy + pgvector. We'll use a tiny **hashing-trick** embedding so the lab stays dependency-free. Swap it for OpenAI or Sentence-Transformers in production — the pattern is identical.

## Learning Objectives

- How to turn text into a vector with pure numpy (the hashing trick)
- **BAD**: Use ANN results directly and hope for the best
- **BETTER**: Over-fetch with ANN, then exact-rerank to boost recall
- **BEST**: Add a domain-aware reranker on top (a stand-in for a cross-encoder)
- How a minimal **RAG** retrieval loop looks end-to-end


## 🛠️ Setup

```bash
cd 03-technologies/databases/vector-databases
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".


In [ ]:
import psycopg2
import numpy as np
import time
import re

DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "vector_demo",
    "user": "demo",
    "password": "demo",
}

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

conn = get_conn()
cur = conn.cursor()
cur.execute("SELECT 1")
conn.close()
print("✅ Connected to PostgreSQL")


## 🔤 Step 1 — Turn Text into a Vector (No API Needed)

Real production code would call something like:

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
vec = model.encode("wireless headphones")  # → 384-dim float32
```

or OpenAI's `text-embedding-3-small`. Both require a download / API key.

For a **self-contained lab**, we'll use the **hashing trick**: chop text into character n-grams, hash each one into a bucket, and use the bucket-count histogram as the vector. It's not as good as a neural embedding but it captures surface similarity — and it runs in a few lines of numpy. The retrieval/rerank/RAG patterns we'll build on top are identical to what you'd use with a real model.


In [ ]:
# Tiny deterministic text → vector: character n-gram hashing trick

DIM = 128  # pgvector column is vector(128), reuse that

def ngrams(text, n=3):
    text = re.sub(r"[^a-z0-9 ]+", " ", text.lower())
    text = re.sub(r"\s+", " ", text).strip()
    padded = f" {text} "
    return [padded[i:i+n] for i in range(len(padded) - n + 1)]

def embed(text, dim=DIM):
    """Hash each n-gram into one of `dim` buckets with a sign flip, then L2-normalize."""
    vec = np.zeros(dim, dtype=np.float32)
    for g in ngrams(text, n=3):
        h = hash(g) % dim
        sign = 1.0 if (hash(g + "_sign") & 1) else -1.0
        vec[h] += sign
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec /= norm
    return vec

def to_pgvector(vec):
    return "[" + ",".join(f"{v:.6f}" for v in vec) + "]"

a = embed("wireless bluetooth headphones")
b = embed("bluetooth wireless earbuds")
c = embed("leather hiking boots")

def cos(x, y): return float(np.dot(x, y))  # vectors already normalized
print(f"wireless bluetooth headphones  vs  bluetooth wireless earbuds : {cos(a,b):.3f}  (should be high)")
print(f"wireless bluetooth headphones  vs  leather hiking boots       : {cos(a,c):.3f}  (should be low)")
print()
print("💡 Similar text → similar vectors. That is the only property we need.")
print("   A neural model would give far better similarity, but the hashing trick")
print("   is enough to demonstrate every retrieval pattern below.")


## 📚 Step 2 — Build a Small Document Corpus

For RAG we need real, readable passages. Let's create a tiny "knowledge base" about system-design topics, embed each passage, and store it in pgvector with an HNSW index.


In [ ]:
CORPUS = [
    ("cache-aside", "In the cache-aside pattern, the application reads from the cache first. On a miss, it loads from the database and populates the cache. Writes go to the database and invalidate or update the cache."),
    ("write-through", "Write-through caching writes to the cache and the database in the same operation. Reads are always fresh, but writes are slower because they must hit both stores."),
    ("write-behind", "Write-behind caching writes to the cache immediately and asynchronously flushes to the database. Very fast writes, but risk of data loss if the cache crashes before a flush."),
    ("consistent-hashing", "Consistent hashing maps keys and nodes onto a ring. When a node is added or removed, only the keys near that node in the ring need to move, which makes it ideal for distributed caches and databases."),
    ("bloom-filter", "A Bloom filter is a space-efficient probabilistic data structure that tests whether an element is in a set. It can return false positives but never false negatives, making it great for avoiding unnecessary disk reads in a database."),
    ("hnsw", "HNSW is a graph-based approximate nearest neighbor index. It builds a hierarchical small-world graph so queries can greedily descend from coarse to fine layers, achieving very high recall with sub-millisecond latency."),
    ("ivfflat", "IVFFlat partitions vectors into clusters using k-means. Queries only scan the clusters whose centroids are closest to the query vector, trading a small amount of recall for much faster search."),
    ("saga", "The saga pattern manages long-running distributed transactions as a sequence of local transactions with compensating actions. Each step can roll back the previous ones if something fails."),
    ("cdc", "Change Data Capture streams row-level changes from a database transaction log to downstream consumers. It powers real-time replication, search indexing, and event-driven architectures."),
    ("rate-limiter-token", "A token-bucket rate limiter refills a bucket at a fixed rate. Each request consumes a token; if the bucket is empty the request is rejected or delayed. It allows short bursts while capping the long-run rate."),
    ("rate-limiter-leaky", "A leaky-bucket rate limiter processes requests at a fixed outflow rate regardless of arrival rate. Excess requests queue or are dropped. It smooths bursty traffic into a steady stream."),
    ("cap", "CAP theorem states that a distributed system can provide at most two of Consistency, Availability, and Partition-tolerance. In practice, networks partition, so the real choice is between CP (consistent) and AP (available)."),
    ("gossip", "Gossip protocols spread information peer-to-peer. Each node periodically picks random peers and exchanges state. They converge quickly and tolerate failures, which is why Cassandra and Consul use them."),
    ("mvcc", "Multi-Version Concurrency Control lets readers see a consistent snapshot without blocking writers. Each write creates a new version; old versions are garbage-collected later. PostgreSQL and MySQL InnoDB both use MVCC."),
    ("sharding", "Sharding splits a dataset horizontally across multiple databases, each holding a subset of rows. A shard key determines routing. Good shard keys distribute load evenly and keep related data on the same shard."),
]

conn = get_conn()
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS kb_docs")
cur.execute("""
    CREATE TABLE kb_docs (
        id SERIAL PRIMARY KEY,
        topic VARCHAR(64) NOT NULL,
        content TEXT NOT NULL,
        embedding vector(128)
    )
""")

for topic, content in CORPUS:
    vec = embed(topic + " " + content)
    cur.execute(
        "INSERT INTO kb_docs (topic, content, embedding) VALUES (%s, %s, %s::vector)",
        (topic, content, to_pgvector(vec)),
    )

# Build an HNSW index using cosine ops (our vectors are normalized)
cur.execute("""
    CREATE INDEX IF NOT EXISTS idx_kb_docs_hnsw
    ON kb_docs USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64)
""")

conn.commit()
cur.execute("SELECT COUNT(*) FROM kb_docs")
print(f"📚 Knowledge base has {cur.fetchone()[0]} passages, HNSW index built.")
conn.close()


## ❌ BAD — Trust the ANN Top-K Blindly

Approximate nearest neighbor indexes (IVFFlat, HNSW) are fast but **approximate**. At default settings, HNSW misses a few true neighbors. If you ask for top-10 and hand them straight to your user (or LLM), you're sometimes showing sub-optimal results.

Let's run a query and see what HNSW returns directly.


In [ ]:
QUERY = "how do I avoid unnecessary database reads for keys that might not exist?"
qvec = embed(QUERY)

conn = get_conn()
cur = conn.cursor()

# Keep ef_search low on purpose — a realistic "default" setting
cur.execute("SET hnsw.ef_search = 10")

cur.execute("""
    SELECT topic, content, embedding <=> %s::vector AS distance
    FROM kb_docs
    ORDER BY embedding <=> %s::vector
    LIMIT 5
""", (to_pgvector(qvec), to_pgvector(qvec)))

print(f"❓ Query: {QUERY!r}")
print("❌ BAD: HNSW top-5, shown to the user directly (ef_search=10)")
print("=" * 80)
for topic, content, dist in cur.fetchall():
    print(f"  [{topic:<22}] dist={dist:.4f}")
    print(f"    {content[:90]}...")
conn.close()

print()
print("⚠️  With a tight ef_search, HNSW can miss the true best neighbor.")
print("   In a RAG system that means the LLM gets the wrong context.")


## ✅ BETTER — Over-Fetch, Then Exact Re-Rank

The classic fix: ask the ANN index for **more candidates than you need** (e.g. top 50 instead of top 5), then do an **exact** distance computation on that small set and keep the true top-5.

This is cheap (scoring 50 vectors is nothing) and typically pushes recall from ~90% to ~100%.


In [ ]:
conn = get_conn()
cur = conn.cursor()

K_FINAL = 5
N_CANDIDATES = 50

cur.execute("SET hnsw.ef_search = 10")  # same loose setting as before

# Step 1: ANN fetches N_CANDIDATES
cur.execute("""
    SELECT id, topic, content, embedding
    FROM kb_docs
    ORDER BY embedding <=> %s::vector
    LIMIT %s
""", (to_pgvector(qvec), N_CANDIDATES))

candidates = cur.fetchall()
print(f"Step 1: HNSW returned {len(candidates)} candidates")

def parse_pgvec(s):
    return np.array([float(x) for x in s.strip("[]").split(",")], dtype=np.float32)

# Step 2: exact re-rank in Python (could also be done in SQL with a CTE)
scored = []
for doc_id, topic, content, emb in candidates:
    doc_vec = parse_pgvec(emb)
    exact_dist = 1.0 - float(np.dot(qvec, doc_vec))  # vectors normalized → cos dist
    scored.append((exact_dist, topic, content))

scored.sort(key=lambda x: x[0])
top = scored[:K_FINAL]

print(f"Step 2: exact rerank → keep top {K_FINAL}\n")
print(f"❓ Query: {QUERY!r}")
print("✅ BETTER: ANN over-fetch (50) → exact rerank → top-5")
print("=" * 80)
for dist, topic, content in top:
    print(f"  [{topic:<22}] dist={dist:.4f}")
    print(f"    {content[:90]}...")
conn.close()

print()
print("💡 Same HNSW index, same ef_search — but recall is effectively 100% because")
print("   we did an exact pass on the small candidate set. This is the single most")
print("   common recall-boosting trick in production vector search.")


## 🏆 BEST — Add a Domain-Aware Reranker

Exact distance is just "how close are the vectors?". Production systems often add a **second reranker** that knows about the domain. Examples:

- A **cross-encoder LLM** that reads the query + each passage and outputs a relevance score (gold standard for RAG).
- **Business-logic boosts**: freshness, author authority, click-through rate.
- **Keyword overlap (BM25)** combined with vector distance.

We'll simulate a reranker with a simple **keyword-overlap boost**: passages that share words with the query get a score bump. This stands in for what a cross-encoder would do, and shows you the architectural pattern.


In [ ]:
STOPWORDS = {"the","a","an","of","for","to","in","on","is","are","i","do",
             "how","what","why","that","might","not","and","or","with","be"}

def keywords(text):
    return {w for w in re.findall(r"[a-z0-9]+", text.lower())
            if w not in STOPWORDS and len(w) > 2}

query_kw = keywords(QUERY)

def rerank_score(exact_dist, content):
    vec_sim = 1.0 - exact_dist
    overlap = len(query_kw & keywords(content)) / max(len(query_kw), 1)
    # weighted combination: vector does the heavy lifting, keyword overlap breaks ties
    return 0.7 * vec_sim + 0.3 * overlap

reranked = []
for dist, topic, content in scored:
    reranked.append((rerank_score(dist, content), topic, content))
reranked.sort(key=lambda x: -x[0])

print(f"❓ Query: {QUERY!r}")
print("🏆 BEST: ANN over-fetch → exact rerank → domain-aware rerank → top-5")
print("=" * 80)
for score, topic, content in reranked[:K_FINAL]:
    print(f"  [{topic:<22}] score={score:.4f}")
    print(f"    {content[:90]}...")

print()
print("💡 The Bloom-filter passage should now rank near the top — it's semantically")
print("   close AND shares keywords like 'database', 'reads', 'exist'. Pure vector")
print("   distance could not see that overlap; the second-stage reranker did.")


## 🤖 Putting It Together: A Minimal RAG Loop

Everything we just built **is** the retrieval half of a RAG system. The "generation" half would hand the top passages to an LLM with a prompt like *"Answer the user's question using only the context below."*

We don't have an LLM in this lab, so we'll stub it — but notice how little glue code it takes once retrieval is solid.


In [ ]:
def retrieve(query, k=3, n_candidates=30):
    qv = embed(query)
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SET hnsw.ef_search = 20")
    cur.execute("""
        SELECT id, topic, content, embedding
        FROM kb_docs
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """, (to_pgvector(qv), n_candidates))
    cands = cur.fetchall()
    conn.close()

    qkw = keywords(query)
    out = []
    for _, topic, content, emb in cands:
        dv = parse_pgvec(emb)
        vec_sim = float(np.dot(qv, dv))
        overlap = len(qkw & keywords(content)) / max(len(qkw), 1)
        out.append((0.7 * vec_sim + 0.3 * overlap, topic, content))
    out.sort(key=lambda x: -x[0])
    return out[:k]

def fake_llm_answer(question, passages):
    # In real life: call OpenAI / Anthropic / local model with this prompt.
    context = "\n\n".join(f"[{t}] {c}" for _, t, c in passages)
    return (
        "PROMPT SENT TO LLM:\n"
        "---\n"
        "Use the context below to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

for q in [
    "how do I avoid unnecessary database reads for keys that might not exist?",
    "what data structure survives node failures in a cache cluster?",
    "how do long-running transactions work across microservices?",
]:
    passages = retrieve(q, k=3)
    print("=" * 85)
    print(f"❓ {q}")
    print()
    for score, topic, content in passages:
        print(f"   • [{topic}]  score={score:.3f}")
    print()
    print(fake_llm_answer(q, passages)[:400] + " ...")
    print()


## 🧹 Cleanup


In [ ]:
conn = get_conn()
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS kb_docs")
conn.commit()
conn.close()
print("🧹 Cleaned up kb_docs")


## 📚 Summary

### Key Takeaways

1. **Text → vector** doesn't require a giant model to learn the pattern — the hashing trick is enough to prototype. Swap it for OpenAI / Sentence-Transformers in production.
2. **Raw ANN results are approximate.** For anything user-facing, over-fetch and re-rank.
3. **Re-ranking has two flavors**: exact distance (cheap, fixes ANN errors) and domain-aware (cross-encoder, BM25, business signals — where most of the quality wins live).
4. **RAG is just retrieval + prompting.** Get retrieval right and the LLM does the rest. Get retrieval wrong and no model can save you.

### Interview Tip

> "For a RAG pipeline I'd use pgvector with an HNSW index for first-stage retrieval, over-fetch (say, top 50), then exact-rerank in SQL or Python. If quality still isn't enough I'd add a cross-encoder reranker on the top 20. That three-stage pipeline — ANN → exact rerank → cross-encoder — is how production search systems at Google, Amazon, and most modern RAG stacks are built."

### What's Next

You've now covered: embeddings, indexes, hybrid search, and RAG. The real next step in a production system is **evaluation** — build a small labeled eval set and measure recall@k and end-to-end answer quality every time you change a parameter.
